# PPSimExample

This notebook demonstrates point-process simulation and parameter recovery for a Poisson CIF.

Workflow:
1. Build a sinusoidal stimulus and Poisson CIF.
2. Simulate multiple spike-train realizations.
3. Fit a Poisson GLM and verify recovery against tolerance thresholds.

Reference: DOI `10.1016/j.jneumeth.2012.08.009`, PMID `22981419`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.spikes import SpikeTrain

rng = np.random.default_rng(2026)
print('Seed:', 2026)


In [ ]:
time = np.linspace(0.0, 8.0, 8001)
dt = float(time[1] - time[0])
stimulus = np.sin(2.0 * np.pi * 1.4 * time) + 0.35 * np.cos(2.0 * np.pi * 0.4 * time)
X = stimulus[:, None]

true_intercept = np.log(14.0)
true_beta = np.array([0.50])
model = CIFModel(coefficients=true_beta, intercept=true_intercept, link='poisson')
true_rate = model.evaluate(X)

n_trials = 18
spike_trains = []
count_vectors = []
for _ in range(n_trials):
    spikes = model.simulate_by_thinning(time, X, rng=rng)
    st = SpikeTrain(spike_times=spikes, t_start=float(time[0]), t_end=float(time[-1]))
    t_bins, counts = st.bin_counts(bin_size_s=dt)
    spike_trains.append(st)
    count_vectors.append(counts)

X_bins = np.interp(t_bins, time, stimulus)[:, None]
X_stack = np.tile(X_bins, (n_trials, 1))
y_stack = np.concatenate(count_vectors)

fit = Analysis.fit_glm(X=X_stack, y=y_stack, fit_type='poisson', dt=dt)
est_rate = fit.predict(X)
coef_error = abs(float(fit.coefficients[0]) - float(true_beta[0]))
rel_rate_err = float(np.mean(np.abs(est_rate - true_rate) / np.maximum(true_rate, 1e-9)))

print('Spike trains simulated:', n_trials)
print('Total spikes:', int(sum(st.spike_times.size for st in spike_trains)))
print('Estimated beta:', float(fit.coefficients[0]))
print('Coefficient abs error:', coef_error)
print('Relative rate error:', rel_rate_err)
print('AIC:', fit.aic())
print('BIC:', fit.bic())

assert coef_error < 0.25, f'Coefficient recovery failed: {coef_error}'
assert rel_rate_err < 0.20, f'Rate recovery failed: {rel_rate_err}'


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=False)

axes[0].plot(time, stimulus, color='k', linewidth=1.2)
axes[0].set_title('Driving stimulus')
axes[0].set_ylabel('a.u.')

axes[1].plot(time, true_rate, label='true rate', linewidth=1.3)
axes[1].plot(time, est_rate, label='estimated rate', linewidth=1.1)
axes[1].set_title('Poisson CIF fit')
axes[1].set_ylabel('Hz')
axes[1].legend(loc='upper right')

for idx, st in enumerate(spike_trains[:10]):
    axes[2].vlines(st.spike_times, idx + 0.6, idx + 1.4, color='k', linewidth=0.6)
axes[2].set_title('Simulated spike rasters (first 10 trials)')
axes[2].set_xlabel('time [s]')
axes[2].set_ylabel('trial')

plt.tight_layout()
plt.show()
